# [Experiment] Vanilla TabNet | Step LR | Forest Cover

### Overview
This notebook establishes the fundamental performance baseline using the standard, unmodified TabNet architecture. The hyperparameters and structural configuration of this baseline have been meticulously aligned to follow the experimental setup detailed in the original TabNet paper as exactly as possible.

### Methodological Context: The Optimization Environment
To create a rigorously grounded control, this baseline employs a standard StepLR learning rate schedule. This specific schedule was selected because it mirrors the exact optimization strategy utilized by the original authors. This discrete, step-wise decay provides a historically validated, rigid optimization trajectory against which all of our subsequent architectural modifications will be measured.

### The Objective
The primary goal of this notebook is to capture the default performance metrics—specifically convergence rate, peak accuracy, and late-stage training stability—of the standard model as originally intended by its creators. This establishes the strict "ground truth" necessary to empirically evaluate the impact of introducing Kolmogorov-Arnold Network (KAN) layers.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier

dataset = fetch_covtype()

X = dataset.data
# CovType is 1-indexed (1 to 7); PyTorch expects 0-indexed labels
y = dataset.target - 1

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape:  {X_test.shape}")

Train shape: (464809, 54)
Valid shape: (58101, 54)
Test shape:  (58102, 54)


### Model Training

In [5]:
MAX_EPOCHS = 1000

In [6]:
# Hyperparameters from original paper.
# Notes:
# - momentum is 1 - 0.7 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 18 epochs approximates the paper's 500 iterations
#   (based on a batch size of 16384 and ~464k training samples).
paper_config = {
    'n_d': 64,
    'n_a': 64,
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-4,
    'momentum': 0.3,
    'optimizer_fn': torch.optim.Adam,
    'optimizer_params': dict(lr=0.02),
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=18, gamma=0.95),
    'mask_type': 'sparsemax'
}

clf_vanilla = TabNetClassifier(
    **paper_config,
    clip_value=2.,
    use_kan=False,
    seed=seed,
    verbose=50
)

In [7]:
# Hyperparameters from original paper.
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_vanilla.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=100,
    num_workers=0,
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 1.15226 | valid_accuracy: 0.03418 |  0:00:08s
epoch 50 | loss: 0.37328 | valid_accuracy: 0.86733 |  0:05:51s
epoch 100| loss: 0.181   | valid_accuracy: 0.92694 |  0:11:34s
epoch 150| loss: 0.13679 | valid_accuracy: 0.94439 |  0:17:18s
epoch 200| loss: 0.0959  | valid_accuracy: 0.95699 |  0:23:01s
epoch 250| loss: 0.07923 | valid_accuracy: 0.96084 |  0:28:44s
epoch 300| loss: 0.07056 | valid_accuracy: 0.96399 |  0:34:30s
epoch 350| loss: 0.06331 | valid_accuracy: 0.96389 |  0:40:15s
epoch 400| loss: 0.0576  | valid_accuracy: 0.96506 |  0:46:02s
epoch 450| loss: 0.05224 | valid_accuracy: 0.96609 |  0:51:50s
epoch 500| loss: 0.0505  | valid_accuracy: 0.96689 |  0:57:36s
epoch 550| loss: 0.04687 | valid_accuracy: 0.96742 |  1:03:23s
epoch 600| loss: 0.04366 | valid_accuracy: 0.96802 |  1:09:10s

Early stopping occurred at epoch 647 with best_epoch = 547 and best_valid_accuracy = 0.96886


In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_vanilla.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 470,580


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_vanilla.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.96955


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/forest_cover/tables'
MODELS_DIR = f'{BASE_DIR}/results/forest_cover/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "Vanilla TabNet",
    "dataset": "Forest Cover",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_vanilla.best_epoch,
    "training_history": clf_vanilla.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/01_vanilla_baseline_step_lr_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_vanilla.save_model(f'{MODELS_DIR}/01_vanilla_baseline_step_lr_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/forest_cover/tables/01_vanilla_baseline_step_lr_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/forest_cover/models/01_vanilla_baseline_step_lr_470k.zip
